# Run CESM-SMYLE benchmark preprocessing

This notebook is a lightweight driver for `run_process_cesm_smyle_benchmark.py`.

It passes the required settings to the preprocessing module and calls `process_one(...)` for selected fields, initialization months, years, ensemble members, and output frequencies.

Typical use cases:

- monthly benchmark for Niño3.4 SST skill: `fields = ["TS"]`, `init_months = [5, 11]`, `freqs = ["mon"]`
- seasonal benchmark for skill maps: `fields = ["TREFHT", "PRECT", "PSL"]`, `init_months = [5, 11]`, `freqs = ["seas"]`
- both monthly and seasonal benchmark files: `freqs = ["mon", "seas"]`

Generated benchmark files are written under the clean model-first diagnostic layout:

- `<S2D_DIAG_ROOT>/CESM-SMYLE/leadtime_acc/inputs/atm/<FIELD>/`
- verification figures, when enabled, go to `/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/CESM-SMYLE/verify/`


The driver validates every requested output against the exact year, member, lead, and
verification-time contract. Missing data or a failed combination makes the production
run fail after reporting all attempted combinations. Existing compatible products are
reused unless `force=True`.


In [7]:
# Basic imports
import os
import sys
import time
import traceback
import warnings
from pathlib import Path

import numpy as np
import xarray as xr

# Optional: Dask for parallel preprocessing
import dask

## 1. User settings

Edit this block for the dataset/variable you want to process.


In [8]:
import sys
from pathlib import Path

# Resolve the active checkout when Jupyter starts in either the repository root
# or its jupyter/ directory. Add it to sys.path so the preprocessing script uses
# this checkout rather than an unrelated installed esp_lab version.
_working_directory = Path.cwd().resolve()
_repo_candidates = (_working_directory, _working_directory.parent)
REPO_ROOT = next(
    (candidate for candidate in _repo_candidates
     if (candidate / "scripts" / "run_process_cesm_smyle_benchmark.py").is_file()
     and (candidate / "esp_lab").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the ESP-Lab repository root or its jupyter directory."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
script_path = REPO_ROOT / "scripts" / "run_process_cesm_smyle_benchmark.py"

# Input/output paths
data_dir = "/global/cfs/cdirs/e3sm/S2S2D/CESM-SMYLE"
outdir = "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE"
figdir = "/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag"

# Variables and initialization months
# For Niño3.4 SST monthly skill, use TS.
fields = ["TS", "TREFHT", "PRECT", "PSL"]

# Usually May and November for the E3SM S2D comparison.
init_months = [2, 5, 8, 11]

# Choose benchmark frequencies to write.
#   "mon"  = monthly benchmark, needed for monthly Niño3.4 bottom panel
#   "seas" = seasonal benchmark, needed for seasonal skill maps/top panels
freqs = ["mon", "seas"]

# Years, members, and lead months
year_start = 1980
year_end = 2018
nens = 20
nlead = 24

# Processing behavior
force = False
dry_run = False
require_all_members = True
verify_coverage = True
open_parallel = False
run_verify = False
close_resources_after_run = True

# Dask settings
use_dask = True
cluster_type = "local"
workers = 4
memory_limit = "4GB"

print("script_path:", script_path)
print("fields:", fields)
print("init_months:", init_months)
print("freqs:", freqs)

script_path: /global/u2/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py
fields: ['TS', 'TREFHT', 'PRECT', 'PSL']
init_months: [2, 5, 8, 11]
freqs: ['mon', 'seas']


## 2. Import the preprocessing module

This imports the `.py` script as a module, so we can call `process_one(...)` directly from the notebook.


In [9]:
import importlib.util

if not script_path.exists():
    raise FileNotFoundError(
        f"Cannot find preprocessing script: {script_path}\n"
        "The repository checkout is incomplete or esp_lab resolves to the wrong checkout."
    )

spec = importlib.util.spec_from_file_location("run_process_cesm_smyle_benchmark", script_path)
prep = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prep)

if outdir is None:
    outdir = prep.OUTDIR_DEFAULT
if figdir is None:
    figdir = prep.FIGDIR_DEFAULT

print("Imported module from:", script_path)
print("Benchmark outdir:", outdir)

Imported module from: /global/u2/z/zhan391/code/ESP-Lab/scripts/run_process_cesm_smyle_benchmark.py
Benchmark outdir: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE


## 3. Start Dask client

The preprocessing module uses Dask/xarray internally. Starting a local Dask client here helps parallelize loading, seasonal aggregation, and NetCDF writing.


In [10]:
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources


def create_benchmark_cluster():
    if not use_dask:
        return None, None
    return get_cluster_client(DaskConfig(
        cluster_type=cluster_type,
        workers=workers,
        memory_limit=memory_limit,
    ))


cluster, client, workflow_resources = restart_notebook_cluster(
    globals(), create_benchmark_cluster,
)
if client is not None:
    print(client)



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
This can easily exceed memory/CPU limits and crash your Jupyter kernel.
We highly recommend launching Jupyter on an Exclusive Perlmutter Compute Node.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43491 instead
  warnings.warn(


Scheduler: tcp://127.0.0.1:46333
Connected workers: 4
Dashboard: http://127.0.0.1:43491/status
<Client: 'tcp://127.0.0.1:46333' processes=4 threads=4, memory=14.90 GiB>


## 4. Build processing list

This mirrors the CLI logic from the `.py` script but keeps all settings visible in the notebook.


In [11]:
allowed_fields = {"TS", "TREFHT", "PRECT", "PSL"}
allowed_freqs = {"mon", "seas"}
if not fields or len(set(fields)) != len(fields) or not set(fields) <= allowed_fields:
    raise ValueError(f"fields must be unique and selected from {sorted(allowed_fields)}")
if not init_months or len(set(init_months)) != len(init_months) or any(
    isinstance(month, bool) or not isinstance(month, int) or not 1 <= month <= 12
    for month in init_months
):
    raise ValueError("init_months must contain unique integer calendar months 1-12")
if not freqs or len(set(freqs)) != len(freqs) or not set(freqs) <= allowed_freqs:
    raise ValueError(f"freqs must be unique and selected from {sorted(allowed_freqs)}")
if year_start > year_end:
    raise ValueError("year_start must be no later than year_end")
if isinstance(nens, bool) or not isinstance(nens, int) or nens < 1:
    raise ValueError("nens must be a positive integer")
if isinstance(nlead, bool) or not isinstance(nlead, int) or nlead < 1:
    raise ValueError("nlead must be a positive integer")
if "seas" in freqs and nlead < 3:
    raise ValueError("seasonal output requires at least three monthly leads")
if use_dask and (workers < 1 or not memory_limit):
    raise ValueError("Dask workers and memory_limit must be positive/non-empty")

Path(outdir).mkdir(parents=True, exist_ok=True)
years = list(range(year_start, year_end + 1))
members = [f"EN{i:02d}" for i in range(1, nens + 1)]
combos = [(field, init_month) for field in fields for init_month in init_months]

print("=" * 70)
print("CESM-SMYLE benchmark preprocessing from notebook")
print("=" * 70)
print(f"repository    : {REPO_ROOT}")
print(f"data_dir      : {data_dir}")
print(f"outdir        : {outdir}")
print(f"figdir        : {figdir}")
print(f"fields        : {fields}")
print(f"init_months   : {init_months}")
print(f"freqs         : {freqs}")
print(f"years         : {years[0]}–{years[-1]}  ({len(years)} years)")
print(f"members       : EN01–EN{nens:02d}  ({nens} members)")
print(f"nlead         : {nlead} months")
print(f"combinations  : {len(combos)}")
print(f"force         : {force}")
print(f"dry_run       : {dry_run}")
print(f"open_parallel : {open_parallel}")
print(f"run_verify    : {run_verify}")
print(f"use_dask      : {use_dask}")
print("=" * 70)


CESM-SMYLE benchmark preprocessing from notebook
repository    : /global/u2/z/zhan391/code/ESP-Lab
data_dir      : /global/cfs/cdirs/e3sm/S2S2D/CESM-SMYLE
outdir        : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE
figdir        : /global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag
fields        : ['TS', 'TREFHT', 'PRECT', 'PSL']
init_months   : [2, 5, 8, 11]
freqs         : ['mon', 'seas']
years         : 1980–2018  (39 years)
members       : EN01–EN20  (20 members)
nlead         : 24 months
combinations  : 16
force         : False
dry_run       : False
open_parallel : False
run_verify    : False
use_dask      : True


## 5. Run preprocessing

This calls:

```python
prep.process_one(...)
```

for each `(field, init_month)` combination.


In [12]:
%%time

total_t0 = time.perf_counter()
counters = {"ok": 0, "skipped": 0, "dry_run": 0, "no_data": 0, "failed": 0}
results = []
interrupted = False

try:
    for field, init_month in combos:
        print("\n" + "-" * 70)
        print(f"Processing field={field}, init_month={init_month:02d}, freqs={freqs}")
        print("-" * 70)
        try:
            status = prep.process_one(
                field=field,
                init_month=init_month,
                data_dir=data_dir,
                outdir=outdir,
                years=years,
                members=members,
                nlead=nlead,
                require_all_members=require_all_members,
                verify_coverage=verify_coverage,
                open_parallel=open_parallel,
                force=force,
                dry_run=dry_run,
                run_verify=run_verify,
                freqs=freqs,
                figdir=figdir,
            )
        except KeyboardInterrupt:
            interrupted = True
            results.append({
                "field": field, "init_month": init_month,
                "freqs": ",".join(freqs), "status": "interrupted",
            })
            break
        except Exception as exc:
            traceback.print_exc()
            warnings.warn(f"FAILED: field={field}, init_month={init_month}: {exc}")
            status = "failed"

        counters[status] = counters.get(status, 0) + 1
        results.append({
            "field": field, "init_month": init_month,
            "freqs": ",".join(freqs), "status": status,
        })
finally:
    elapsed = time.perf_counter() - total_t0
    print("\n" + "=" * 70)
    print(f"Finished in {elapsed:.1f}s")
    for status in ("ok", "skipped", "dry_run", "no_data", "failed"):
        print(f"{status:10s}: {counters.get(status, 0)}")
    print("=" * 70)
    if close_resources_after_run:
        close_notebook_resources(globals())
        print("Closed notebook Dask resources.")

if interrupted:
    raise KeyboardInterrupt("CESM-SMYLE benchmark preprocessing was interrupted")
unsuccessful = [result for result in results if result["status"] in {"failed", "no_data"}]
if unsuccessful:
    details = ", ".join(
        f"{item['field']}/init{item['init_month']:02d}:{item['status']}"
        for item in unsuccessful
    )
    raise RuntimeError(f"Benchmark preprocessing did not complete successfully: {details}")
if len(results) != len(combos):
    raise RuntimeError(f"Processed {len(results)} of {len(combos)} requested combinations")



----------------------------------------------------------------------
Processing field=TS, init_month=02, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TS, init_month=05, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TS, init_month=08, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TS, init_month=11, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=TREFHT, init_month=02, freqs=['mon', 'seas']
--------------------------------------------------------------

  [REBUILD] invalid existing benchmark: BSMYLE02_PSL_N20_M24_seas.nc (initialization cohort differs (Y=38, expected 39; missing=['1999020100'], extra=[]))



----------------------------------------------------------------------
Processing field=PSL, init_month=02, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=PSL, init_month=05, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=PSL, init_month=08, freqs=['mon', 'seas']
----------------------------------------------------------------------

----------------------------------------------------------------------
Processing field=PSL, init_month=11, freqs=['mon', 'seas']
----------------------------------------------------------------------

Finished in 504.9s
ok        : 1
skipped   : 15
dry_run   : 0
no_data   : 0
failed    : 0
Closed notebook Dask resources.
CPU times: user 2min 53s, sys: 35.5 s, total: 3min 29s
Wall time: 8mi

## 6. Review outputs

This cell lists expected output files using the module's benchmark filename convention.


In [13]:
expected_files = []
invalid_outputs = []

for field in fields:
    for init_month in init_months:
        for freq in freqs:
            fname = prep.benchmark_filename(
                field, init_month, nens=nens, nlead=nlead, freq=freq,
            )
            path = Path(outdir) / "leadtime_acc" / "inputs" / "atm" / field / fname
            expected_files.append(path)
            if not path.exists():
                invalid_outputs.append(f"{path}: missing")
                print(f"[MISSING] {path}")
                continue
            issues = prep.existing_benchmark_issues(
                path, field=field, init_month=init_month,
                years=years, members=members, nlead=nlead, freq=freq,
            )
            if issues:
                invalid_outputs.append(f"{path}: {'; '.join(issues)}")
                print(f"[INVALID] {path}: {'; '.join(issues)}")
            else:
                size_gb = path.stat().st_size / 1024**3
                print(f"[VALID]   {path}  ({size_gb:.2f} GB)")

if invalid_outputs and not dry_run:
    raise RuntimeError(
        "Benchmark output validation failed:\n  - " + "\n  - ".join(invalid_outputs)
    )
if not invalid_outputs:
    print(f"Validated all {len(expected_files)} requested benchmark files.")


[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE02_TS_N20_M24_mon.nc  (2.15 GB)
[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE02_TS_N20_M24_seas.nc  (0.63 GB)
[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE05_TS_N20_M24_mon.nc  (2.15 GB)
[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE05_TS_N20_M24_seas.nc  (0.63 GB)
[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE08_TS_N20_M24_mon.nc  (2.15 GB)
[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE08_TS_N20_M24_seas.nc  (0.63 GB)
[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE11_TS_N20_M24_mon.nc  (2.15 GB)
[VALID]   /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE/leadtime_acc/inputs/atm/TS/BSMYLE11_TS_N20_M24_seas.nc  (0.63 GB)
[VAL

## 7. Close Dask resources

The processing cell closes resources automatically by default. This cell is idempotent
and can also be run after an interrupted cell or when `close_resources_after_run=False`.


In [ ]:
close_notebook_resources(globals())
print('Closed notebook Dask resources.')
